In [84]:
import os
os.chdir(r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10')

In [86]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [87]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [88]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [89]:
model1 = MLP()
model1.summary()

c:\Users\Muhammad Mubien\Documents\python env 3.12\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 504)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │        16,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,193 (63.25 KB)

 Trainable params: 16,193 (63.25 KB)

 Non-trainable params: 0 (0.00 B)

In [90]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [91]:
checkpoints = r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

<>:3: SyntaxWarning: invalid escape sequence '\h'
<>:4: SyntaxWarning: invalid escape sequence '\h'
<>:3: SyntaxWarning: invalid escape sequence '\h'
<>:4: SyntaxWarning: invalid escape sequence '\h'
C:\Users\Muhammad Mubien\AppData\Local\Temp\ipykernel_14516\181276577.py:3: SyntaxWarning: invalid escape sequence '\h'
  FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
C:\Users\Muhammad Mubien\AppData\Local\Temp\ipykernel_14516\181276577.py:4: SyntaxWarning: invalid escape sequence '\h'
  JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])


In [92]:
os.path.exists(JSON_PATH)

True

In [93]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [94]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(getattr(model.optimizer, "lr", model.optimizer.learning_rate))))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(getattr(model.optimizer, "lr", model.optimizer.learning_rate))))

[INFO] compiling model...


In [43]:
import os
path_dataset =r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEPscaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Muhammad Mubien\Documents\python env 3.12\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [95]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.009993553161621094 sec


In [96]:
train_X.shape

(835, 24, 21)

In [97]:
epochs = 5
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/5
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3586 - mae: 0.3586 - mape: 180.4849
Epoch 1: val_loss improved from None to 0.14279, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0001-loss0.14.h5



Epoch 1: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0001-loss0.14.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - loss: 0.2714 - mae: 0.2714 - mape: 140.3583 - val_loss: 0.1428 - val_mae: 0.1428 - val_mape: 44.9272
Epoch 2/5
22/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1539 - mae: 0.1539 - mape: 80.3410  
Epoch 2: val_loss did not improve from 0.14279
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.1279 - mae: 0.1279 - mape: 61.2539 - val_loss: 0.1921 - val_mae: 0.1921 - val_mape: 66.2048
Epoch 3/5
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1459 - mae: 0.1459 - mape: 84.1988 
Epoch 3: val_loss improved from 0.14279 to 0.10143, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0003-loss0.10.h5



Epoch 3: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0003-loss0.10.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.1414 - mae: 0.1414 - mape: 81.0886 - val_loss: 0.1014 - val_mae: 0.1014 - val_mape: 37.6794
Epoch 4/5
12/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1103 - mae: 0.1103 - mape: 47.8284 
Epoch 4: val_loss did not improve from 0.10143
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1000 - mae: 0.1000 - mape: 46.3587 - val_loss: 0.1018 - val_mae: 0.1018 - val_mape: 34.7526
Epoch 5/5
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0802 - mae: 0.0802 - mape: 36.4342 
Epoch 5: val_loss improved from 0.10143 to 0.04488, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0005-loss0.04.h5



Epoch 5: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0005-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0815 - mae: 0.0815 - mape: 42.0013 - val_loss: 0.0449 - val_mae: 0.0449 - val_mape: 14.0995


In [100]:
model = load_model(
    r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0001-loss0.14.h5',
    compile=False
)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
Mean Absolute Error (MAE): 2341.5
Median Absolute Error (MedAE): 1889.49
Mean Squared Error (MSE): 5994743.39
Root Mean Squared Error (RMSE): 2448.42
Mean Absolute Percentage Error (MAPE): 14.9 %
Median Absolute Percentage Error (MDAPE): 12.21 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# Fine Tuning

In [111]:
checkpoints = r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0001-loss0.14.h5'
start_epoch= 56

In [122]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
model = load_model(
    r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0001-loss0.14.h5',
    compile=False
)
    # update the learning rate
opt = Adam(1e-3)
model.compile(loss='mae', optimizer=opt, metrics=["mae", "mape"])
y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
    

[INFO] loading <Sequential name=sequential_3, built=True>...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step


In [123]:
epochs = 5
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/5
17/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2860 - mae: 0.2860 - mape: 113.2887
Epoch 1: val_loss improved from None to 0.08356, saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E2-cp-0001-loss0.08.h5



Epoch 1: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E2-cp-0001-loss0.08.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 0.2116 - mae: 0.2116 - mape: 102.2031 - val_loss: 0.0836 - val_mae: 0.0836 - val_mape: 29.9839
Epoch 2/5
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1159 - mae: 0.1159 - mape: 57.6628
Epoch 2: val_loss did not improve from 0.08356
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0983 - mae: 0.0983 - mape: 52.6057 - val_loss: 0.1213 - val_mae: 0.1213 - val_mape: 44.3556
Epoch 3/5
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0747 - mae: 0.0747 - mape: 34.5853 
Epoch 3: val_loss did not improve from 0.08356
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0763 - mae: 0.0763 - mape: 39.2913 - val_loss: 0.0841 - val_mae: 0.0841 - val_mape: 29.2918
Epoch 4/5
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0818 - mae: 0.0818 - mape: 43.9309
Epoch 4: val_loss did not improve from 0.08356
27/27 ━━━━━━━━━━━━━━━━━━━━ 


Epoch 5: finished saving model to C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E2-cp-0005-loss0.07.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0733 - mae: 0.0733 - mape: 37.1326 - val_loss: 0.0706 - val_mae: 0.0706 - val_mape: 24.2827


In [125]:

model = load_model(
    r'C:\Users\Muhammad Mubien\Documents\python env 3.12\lab10\delete\E1-cp-0001-loss0.14.h5',
    compile=False
)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
Mean Absolute Error (MAE): 2341.5
Median Absolute Error (MedAE): 1889.49
Mean Squared Error (MSE): 5994743.39
Root Mean Squared Error (RMSE): 2448.42
Mean Absolute Percentage Error (MAPE): 14.9 %
Median Absolute Percentage Error (MDAPE): 12.21 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)
